# Implementation !

Since we can't train a transformer over text here, simply because it would require huge datasets and computation resources, let's try some easier case that tries to mimick what happens in text.
Let's suppose a vocabulary consists of only three words, or tokens : a quare, a circle, and a cross $ \Omega = \{ X, O, \Box \}$

Now we must naturally introduce some rules for this language, let them be the following : 
- we can rither repeate one token, in this sentence no other word must intrefere : Ex : "X X X X X X"
- we can either have doubling sequence, repeating only two tokens : Ex : " X O X O X O"
- and finally, a circular rotation of all three tokens : " X O sq X O sq X O sq "

In [1]:
# ── Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [2]:
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


In [3]:
# ── Vocabulary ───────────────────────────────────────────────────────────────
# x=0  o=1  sq=2  <PAD>=3
VOCAB      = {"x": 0, "o": 1, "sq": 2, "<PAD>": 3}
VOCAB_SIZE = len(VOCAB)
IDX2TOK    = {v: k for k, v in VOCAB.items()}
X, O, SQ   = 0, 1, 2

In [ ]:
# ── Pattern generators ───────────────────────────────────────────────────────
def pattern_repeat(token, length=12):
    """x x x x x x"""
    return [token] * length

def pattern_cycle3(length=12):
    """x o sq x o sq ..."""
    base = [X, O, SQ]
    return (base * (length // 3 + 1))[:length]

def pattern_alternate1(length=12):
    """x o x o x o ..."""
    base = [X, O]
    return (base * (length // 2 + 1))[:length]

def pattern_alternate2(length=12):
    """x o x o x o ..."""
    base = [X, SQ]
    return (base * (length // 2 + 1))[:length]


PATTERNS = [
    lambda: pattern_repeat(X),
    lambda: pattern_repeat(O),
    lambda: pattern_repeat(SQ),
    pattern_cycle3,
    pattern_alternate1,
    pattern_alternate2,
]


In [14]:
# ── Dataset ──────────────────────────────────────────────────────────────────
class SequenceDataset(Dataset):
    def __init__(self, n_samples=8000, seq_len=12):
        self.seq_len = seq_len
        self.data    = []
        for _ in range(n_samples):
            fn  = PATTERNS[np.random.randint(len(PATTERNS))]
            seq = fn()
            # input = seq[:-1], target = seq[1:]  (autoregressive)
            self.data.append((
                torch.tensor(seq[:-1], dtype=torch.long),
                torch.tensor(seq[1:],  dtype=torch.long),
            ))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [42]:
dataset    = SequenceDataset(n_samples=8000, seq_len=12)
loader     = DataLoader(dataset, batch_size=128, shuffle=True)

In [ ]:
# ── Transformer ──────────────────────────────────────────────────────────────
class TinyTransformer(nn.Module):
    """
    Small causal transformer for sequence modelling.

    Args:
        vocab_size  (int): Number of tokens.
        embed_dim   (int): Embedding dimensionality.
        n_heads     (int): Number of attention heads.
        n_layers    (int): Number of transformer blocks.
        seq_len     (int): Maximum sequence length.
        dropout     (float): Dropout probability.
    """
    def __init__(
        self,
        vocab_size: int = VOCAB_SIZE,
        embed_dim:  int = 32,
        n_heads:    int = 2,
        n_layers:   int = 2,
        seq_len:    int = 11,
        dropout:    float = 0.1,
    ):
        super().__init__()
        self.embed_dim = embed_dim
        self.seq_len   = seq_len

        # ── Token + positional embeddings ────────────────────────────────────
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb   = nn.Embedding(seq_len,    embed_dim)

        # ── Transformer encoder layers (causal mask applied at forward) ──────
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=64, dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # ── Output projection ────────────────────────────────────────────────
        self.head = nn.Linear(embed_dim, vocab_size)

    def _causal_mask(self, size: int) -> torch.Tensor:
        """Upper-triangular mask to prevent attending to future tokens."""
        return torch.triu(torch.ones(size, size, device=device), diagonal=1).bool()

    def forward(self, x: torch.Tensor):
        B, T    = x.shape
        pos     = torch.arange(T, device=device).unsqueeze(0)           # (1, T)
        emb     = self.token_emb(x) + self.pos_emb(pos)                 # (B, T, D)
        mask    = self._causal_mask(T)
        out     = self.transformer(emb, mask=mask)                      # (B, T, D)
        logits  = self.head(out)                                        # (B, T, V)
        return logits, out   # return hidden states for embedding analysis

    def get_contextual_embeddings(self, x: torch.Tensor):
        """Return hidden states (contextual embeddings) without logits."""
        _, hidden = self.forward(x)
        return hidden